In [ ]:
# forbidden access (i think just set the flag upon super.init)
# because the flag prevents access of the super methods (and i don't think it's as much of a problem in this implementation)
# build dm

In [ ]:
"""
sandbox_time.ipynb

A sandbox to develop a time-resolved class.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""

import numpy as np
import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# SANITY CHECKS
# beta weight sanity checks
# -> multiply beta weight by 1/binwidth_s and make sure identical to firing rate
"""--------------------------------------------"""

## init

In [ ]:
# view_fits
# plot_sctavg_weights
# add interaction terms and trials from block switch

In [ ]:
from sg.models import make_tre, Encoder

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=False,
    stepsize_s=0.1,
)
encoder.verify()

"""
encoder_mb = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=False,
    stepsize_s=0.1,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=False,
    stepsize_s=0.1,
    strategy_filter="mf",
)
"""

In [ ]:
encoder.view_fits()

In [ ]:
encoder.view_peths()

In [ ]:
encoder.robs_predict["baseline"][:: encoder.num_bins, encoder.reg_idxs["DMS"]]
# encoder.robs_tavg[:,encoder.reg_idxs["DMS"]]

In [ ]:
a = np.arange(24).reshape(2, 3, 4)
b = a.transpose(1, 0, 2).reshape(6, 4)
c = b.reshape(3, 2, 4).transpose(1, 0, 2)

a == c

In [ ]:
encoder.view_fits(model="baseline")

In [ ]:
encoder.view_weights(regr="response", peth_mode="response")

In [ ]:
encoder.view_weights()

In [ ]:
unit_idx = 120
regr = "response"

regr_idxs = [encoder.dm_idxs[f"{regr}_{i}"] for i in range(15)]
plt.figure()
plt.plot(encoder.encoder_weights[unit_idx, regr_idxs])
plt.show()

In [ ]:
encoder.robs.shape, encoder.robs_predict["encoder"].shape

In [ ]:
bins = 2
trials = 3
neurons = 4

X = np.arange(24).reshape(bins, trials, neurons)  # 2 bins, 3 trials, 4 neurons
Y = X.transpose(1, 0, 2).reshape(bins * trials, neurons)
Z = Y.reshape(trials, bins, neurons).transpose(1, 0, 2)
assert np.all(X == Z)
X, Y, Z

In [ ]:
from copy import deepcopy
from core.viz import plot_scatter

thing = "response_0"

idx = encoder.dm_idxs[thing] - encoder.num_tents
tv_ko = deepcopy(encoder.tvs)
tv_ko[:, idx] = 0

robs_predict = encoder.encoder.predict(tv_ko)
robs_2_subtract = encoder.robs_predict["baseline"] + robs_predict
robs_corrected = encoder.robs - robs_2_subtract

corr_mask = encoder.trial_data.rewarded == 1
incorr_mask = encoder.trial_data.rewarded == 0

robs_corrected_3d = robs_corrected.reshape(
    encoder.num_trials, encoder.num_bins, encoder.num_units
).transpose(1, 0, 2)

robs_corr = robs_corrected_3d[:, corr_mask, :].mean(axis=(0, 1))
robs_incorr = robs_corrected_3d[:, incorr_mask, :].mean(axis=(0, 1))

bweight_robs = (robs_corr - robs_incorr) / 2

for i in range(80):
    bweight = encoder.encoder_weights[:, encoder.dm_idxs[thing]]

plot_scatter(bweight_robs, bweight)

In [ ]:
import numpy as np

for key in encoder.tv_keys:
    print(key, np.unique(encoder.trial_data[key]))

## the t-population

In [ ]:
# select the neurons that lie along the axis
# plot their encoding for mb/mf
# is it just a different encoding pattern (i.e., they encode different things)
# or are they silent(er) in another strategy